# Ingesta de géneros (incremental)

Lee `genre.csv` (CSV) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Reemplaza la tabla Delta `movie_silver.genres` en cada ejecución (`overwrite`), por lo que conserva solo el último lote procesado.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

### Lectura del archivo

Primero lee `genre.csv` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
genre_df = (
    spark.read
        .format("csv")
        .option("header", True)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/genre.csv")
)

In [0]:
genre_schema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("genreName", StringType(), True)
])
genre_df = (
    spark.read
        .format('csv')
        .option("header", True)
        .schema(genre_schema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/genre.csv")
)

### Transformación de columnas

Renombra `genreId` y `genreName` a snake_case y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
genre_df_final = (
    add_ingestion_date(genre_df)
    .select(
        col("genreId").alias("genre_id"),
        col("genreName").alias("genre_name"),
        lit(v_env).alias("env"),
        lit(v_file_date).alias("file_date")
    )
)

### Escritura en Silver

Reemplaza la tabla Delta `movie_silver.genres` (modo `overwrite`).

In [0]:
(
    genre_df_final.write
        .mode("overwrite")
        .saveAsTable("movie_silver.genres")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")